# Bài 3 — Luật kết hợp (Association Rules)

**Dataset:** KuaiLive Viewer Engagement  
**Thuật toán:** Apriori  
**Mục tiêu:** Tìm các hành vi của người xem thường xuất hiện cùng nhau bằng các độ đo Support, Confidence và Lift.

Mỗi dòng dữ liệu được xem là một giao dịch, đại diện cho hành vi của một viewer trong một phiên xem livestream.

In [3]:
import pandas as pd
import numpy as np

from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Đã import thư viện thành công.")

Đã import thư viện thành công.


In [4]:
# Đường dẫn đến dữ liệu đã tiền xử lý từ Bài 1
file_path = "../data/processed/dataset_30_attributes_processed.csv"

df = pd.read_csv(file_path)

print("Đọc dữ liệu thành công!")
print("Kích thước dữ liệu:", df.shape)

display(df.head())

Đọc dữ liệu thành công!
Kích thước dữ liệu: (20000, 47)


,Viewer_ID,Live_ID,Streamer_ID,Viewer_Age,Viewer_Gender,Viewer_Country,Viewer_Device_Brand,Viewer_Device_Price,Account_Age_Days,Accumulated_Watch_Count,Accumulated_Watch_Duration,Is_Live_Streamer,Is_Photo_Author,Watch_Time,Click_Count,Comment_Count,Like_Count,Gift_Count,Gift_Value,Has_Comment,Has_Like,Has_Gift,Interaction_Count,Like_Rate,Comment_Rate,Gift_Rate,Engagement_Score,Engagement_Level,Streamer_Gender,Streamer_Age,Log_Watch_Time,Log_Click_Count,Log_Comment_Count,Log_Like_Count,Log_Gift_Count,Log_Gift_Value,Z_Watch_Time,Z_Click_Count,Z_Comment_Count,Z_Like_Count,Z_Gift_Count,Z_Gift_Value,Watch_Time_Level,Like_Behavior,Comment_Behavior,Gift_Behavior,Gift_Value_Level
0,1,1520913,285858,24-30,F,China,HUAWEI,1000-2000,872,50000-100000,0-1000000000,1,1,4519,1,0,0,0,0,0,0,0,0,0.000000,0.0,0.0,3.3665,Medium,F,24-30,8.416267,0.693147,0.0,0.000000,0.0,0.0,-0.291782,-0.326134,-0.191158,-0.198031,-0.119668,-0.095052,Medium,No_Like,No_Comment,No_Gift,No_Gift
1,7,541927,244121,18-23,M,China,IQOO,2000-4000,1490,100000-500000,1000000000-5000000000,1,1,1600101,1,0,1,0,0,0,1,0,1,0.000037,0.0,0.0,5.8529,High,M,24-30,14.285578,0.693147,0.0,0.693147,0.0,0.0,2.801102,-0.326134,-0.191158,4.776307,-0.119668,-0.095052,Long,Liked,No_Comment,No_Gift,No_Gift
2,19,5056801,392962,12-17,F,China,VIVO,1000-2000,807,100000-500000,5000000000-10000000000,1,1,11329,1,0,0,0,0,0,0,0,0,0.000000,0.0,0.0,3.7341,Medium,M,18-23,9.335209,0.693147,0.0,0.000000,0.0,0.0,0.192463,-0.326134,-0.191158,-0.198031,-0.119668,-0.095052,Medium,No_Like,No_Comment,No_Gift,No_Gift
3,19,5605873,107543,12-17,F,China,VIVO,1000-2000,807,100000-500000,5000000000-10000000000,1,1,2267,1,0,0,0,0,0,0,0,0,0.000000,0.0,0.0,3.0907,Medium,F,18-23,7.726654,0.693147,0.0,0.000000,0.0,0.0,-0.655180,-0.326134,-0.191158,-0.198031,-0.119668,-0.095052,Medium,No_Like,No_Comment,No_Gift,No_Gift
4,22,1277396,33084,31-40,M,China,HONOR,2000-4000,3220,0-50000,1000000000-5000000000,0,1,26729,1,0,0,0,0,0,0,0,0,0.000000,0.0,0.0,4.0774,High,M,18-23,10.193542,0.693147,0.0,0.000000,0.0,0.0,0.644768,-0.326134,-0.191158,-0.198031,-0.119668,-0.095052,Long,No_Like,No_Comment,No_Gift,No_Gift


In [5]:
# Các thuộc tính dạng rời rạc được tạo từ Bài 1
transaction_cols = [
    "Watch_Time_Level",
    "Like_Behavior",
    "Comment_Behavior",
    "Gift_Behavior",
    "Gift_Value_Level"
]

print("Các thuộc tính dùng cho Association Rules:")
for col in transaction_cols:
    print("-", col)

display(df[transaction_cols].head())

Các thuộc tính dùng cho Association Rules:
- Watch_Time_Level
- Like_Behavior
- Comment_Behavior
- Gift_Behavior
- Gift_Value_Level


,Watch_Time_Level,Like_Behavior,Comment_Behavior,Gift_Behavior,Gift_Value_Level
0,Medium,No_Like,No_Comment,No_Gift,No_Gift
1,Long,Liked,No_Comment,No_Gift,No_Gift
2,Medium,No_Like,No_Comment,No_Gift,No_Gift
3,Medium,No_Like,No_Comment,No_Gift,No_Gift
4,Long,No_Like,No_Comment,No_Gift,No_Gift


In [6]:
# Kiểm tra giá trị thiếu
missing = df[transaction_cols].isnull().sum()

missing_df = pd.DataFrame({
    "Thuộc tính": missing.index,
    "Số giá trị thiếu": missing.values
})

display(missing_df)

print(
    "Tổng giá trị thiếu:",
    df[transaction_cols].isnull().sum().sum()
)

,Thuộc tính,Số giá trị thiếu
0,Watch_Time_Level,0
1,Like_Behavior,0
2,Comment_Behavior,0
3,Gift_Behavior,0
4,Gift_Value_Level,0


Tổng giá trị thiếu: 0


In [7]:
# Phân bố các item
for col in transaction_cols:
    print(f"\nPhân bố {col}:")
    display(
        df[col]
        .value_counts(dropna=False)
        .to_frame("Số lượng")
    )


Phân bố Watch_Time_Level:


,Số lượng
Watch_Time_Level,
Long,6667
Short,6667
Medium,6666



Phân bố Like_Behavior:


,Số lượng
Like_Behavior,
No_Like,19229
Liked,771



Phân bố Comment_Behavior:


,Số lượng
Comment_Behavior,
No_Comment,19279
Commented,721



Phân bố Gift_Behavior:


,Số lượng
Gift_Behavior,
No_Gift,19716
Gifted,284



Phân bố Gift_Value_Level:


,Số lượng
Gift_Value_Level,
No_Gift,19716
Low_Gift,239
High_Gift,45


## 1. Định nghĩa giao dịch

Trong bài này, **mỗi dòng dữ liệu được xem là một giao dịch**, đại diện cho hành vi của một viewer trong một phiên xem livestream.

Các thuộc tính hành vi trong cùng một dòng được xem là các item xuất hiện trong cùng một “giỏ”.

In [8]:
transactions = []

for _, row in df[transaction_cols].iterrows():
    transaction = []

    for col in transaction_cols:
        value = row[col]

        if pd.notna(value):
            transaction.append(f"{col}={value}")

    transactions.append(transaction)

print("Số giao dịch:", len(transactions))

print("\nVí dụ 5 giao dịch đầu:")
for transaction in transactions[:5]:
    print(transaction)

Số giao dịch: 20000

Ví dụ 5 giao dịch đầu:
['Watch_Time_Level=Medium', 'Like_Behavior=No_Like', 'Comment_Behavior=No_Comment', 'Gift_Behavior=No_Gift', 'Gift_Value_Level=No_Gift']
['Watch_Time_Level=Long', 'Like_Behavior=Liked', 'Comment_Behavior=No_Comment', 'Gift_Behavior=No_Gift', 'Gift_Value_Level=No_Gift']
['Watch_Time_Level=Medium', 'Like_Behavior=No_Like', 'Comment_Behavior=No_Comment', 'Gift_Behavior=No_Gift', 'Gift_Value_Level=No_Gift']
['Watch_Time_Level=Medium', 'Like_Behavior=No_Like', 'Comment_Behavior=No_Comment', 'Gift_Behavior=No_Gift', 'Gift_Value_Level=No_Gift']
['Watch_Time_Level=Long', 'Like_Behavior=No_Like', 'Comment_Behavior=No_Comment', 'Gift_Behavior=No_Gift', 'Gift_Value_Level=No_Gift']


## 2. Nhị phân hóa dữ liệu giao dịch

Dữ liệu giao dịch được chuyển sang dạng Boolean để Apriori có thể xác định item nào xuất hiện trong từng giao dịch.

In [9]:
te = TransactionEncoder()

te_array = te.fit(transactions).transform(transactions)

basket = pd.DataFrame(
    te_array,
    columns=te.columns_
)

print("Kích thước dữ liệu sau nhị phân hóa:", basket.shape)

display(basket.head())

Kích thước dữ liệu sau nhị phân hóa: (20000, 12)


,Comment_Behavior=Commented,Comment_Behavior=No_Comment,Gift_Behavior=Gifted,Gift_Behavior=No_Gift,Gift_Value_Level=High_Gift,Gift_Value_Level=Low_Gift,Gift_Value_Level=No_Gift,Like_Behavior=Liked,Like_Behavior=No_Like,Watch_Time_Level=Long,Watch_Time_Level=Medium,Watch_Time_Level=Short
0,False,True,False,True,False,False,True,False,True,False,True,False
1,False,True,False,True,False,False,True,True,False,True,False,False
2,False,True,False,True,False,False,True,False,True,False,True,False
3,False,True,False,True,False,False,True,False,True,False,True,False
4,False,True,False,True,False,False,True,False,True,True,False,False


In [10]:
# Support của từng item
item_support = basket.mean().sort_values(ascending=False)

item_support_df = item_support.reset_index()
item_support_df.columns = ["Item", "Support"]

display(item_support_df.round(4))

,Item,Support
0,Gift_Behavior=No_Gift,0.9858
1,Gift_Value_Level=No_Gift,0.9858
2,Comment_Behavior=No_Comment,0.9640
3,Like_Behavior=No_Like,0.9614
4,Watch_Time_Level=Long,0.3334
5,Watch_Time_Level=Short,0.3334
6,Watch_Time_Level=Medium,0.3333
7,Like_Behavior=Liked,0.0386
8,Comment_Behavior=Commented,0.0360
9,Gift_Behavior=Gifted,0.0142


## 3. Apriori — Cấu hình 1

Chọn `min_support = 0.01` và `min_confidence = 0.30`.

Ngưỡng này được dùng làm cấu hình ban đầu để tìm các itemset phổ biến.

In [11]:
min_support_1 = 0.01
min_confidence = 0.30

frequent_itemsets_1 = apriori(
    basket,
    min_support=min_support_1,
    use_colnames=True
)

frequent_itemsets_1 = frequent_itemsets_1.sort_values(
    by="support",
    ascending=False
).reset_index(drop=True)

print("Min support:", min_support_1)
print("Số frequent itemsets:", len(frequent_itemsets_1))

display(frequent_itemsets_1.head(20))

Min support: 0.01
Số frequent itemsets: 103


,support,itemsets
0,0.98580,frozenset({Gift_Behavior=No_Gift})
1,0.98580,frozenset({Gift_Value_Level=No_Gift})
2,0.98580,"frozenset({Gift_Value_Level=No_Gift, Gift_Beha..."
3,0.96395,frozenset({Comment_Behavior=No_Comment})
4,0.96145,frozenset({Like_Behavior=No_Like})
5,0.95810,"frozenset({Gift_Behavior=No_Gift, Comment_Beha..."
6,0.95810,"frozenset({Gift_Value_Level=No_Gift, Comment_B..."
7,0.95810,"frozenset({Gift_Value_Level=No_Gift, Gift_Beha..."
8,0.95355,"frozenset({Like_Behavior=No_Like, Gift_Behavio..."
9,0.95355,"frozenset({Like_Behavior=No_Like, Gift_Value_L..."


In [12]:
rules_1 = association_rules(
    frequent_itemsets_1,
    metric="confidence",
    min_threshold=min_confidence
)

rules_1 = rules_1.sort_values(
    by="lift",
    ascending=False
).reset_index(drop=True)

print("Số luật tạo được:", len(rules_1))

display(
    rules_1[
        [
            "antecedents",
            "consequents",
            "support",
            "confidence",
            "lift"
        ]
    ].head(20).round(4)
)

Số luật tạo được: 544


,antecedents,consequents,support,confidence,lift
0,"frozenset({Gift_Behavior=Gifted, Watch_Time_Le...",frozenset({Gift_Value_Level=Low_Gift}),0.0118,0.8434,70.5788
1,frozenset({Gift_Value_Level=Low_Gift}),"frozenset({Gift_Behavior=Gifted, Watch_Time_Le...",0.0118,0.9916,70.5788
2,"frozenset({Gift_Value_Level=Low_Gift, Watch_Ti...",frozenset({Gift_Behavior=Gifted}),0.0118,1.0000,70.4225
3,frozenset({Gift_Behavior=Gifted}),"frozenset({Gift_Value_Level=Low_Gift, Watch_Ti...",0.0118,0.8345,70.4225
4,frozenset({Gift_Value_Level=Low_Gift}),frozenset({Gift_Behavior=Gifted}),0.0120,1.0000,70.4225
5,frozenset({Gift_Behavior=Gifted}),frozenset({Gift_Value_Level=Low_Gift}),0.0120,0.8415,70.4225
6,"frozenset({Like_Behavior=Liked, Watch_Time_Lev...",frozenset({Comment_Behavior=Commented}),0.0136,0.3810,10.5673
7,frozenset({Comment_Behavior=Commented}),"frozenset({Like_Behavior=Liked, Watch_Time_Lev...",0.0136,0.3773,10.5673
8,"frozenset({Watch_Time_Level=Long, Comment_Beha...",frozenset({Like_Behavior=Liked}),0.0136,0.3836,9.9517
9,frozenset({Like_Behavior=Liked}),"frozenset({Watch_Time_Level=Long, Comment_Beha...",0.0136,0.3528,9.9517


## 4. Apriori — Cấu hình 2

Tăng `min_support` lên `0.03` để so sánh với cấu hình 1.

In [13]:
min_support_2 = 0.03

frequent_itemsets_2 = apriori(
    basket,
    min_support=min_support_2,
    use_colnames=True
)

frequent_itemsets_2 = frequent_itemsets_2.sort_values(
    by="support",
    ascending=False
).reset_index(drop=True)

print("Min support:", min_support_2)
print("Số frequent itemsets:", len(frequent_itemsets_2))

display(frequent_itemsets_2.head(20))

Min support: 0.03
Số frequent itemsets: 70


,support,itemsets
0,0.98580,frozenset({Gift_Value_Level=No_Gift})
1,0.98580,frozenset({Gift_Behavior=No_Gift})
2,0.98580,"frozenset({Gift_Value_Level=No_Gift, Gift_Beha..."
3,0.96395,frozenset({Comment_Behavior=No_Comment})
4,0.96145,frozenset({Like_Behavior=No_Like})
5,0.95810,"frozenset({Gift_Value_Level=No_Gift, Comment_B..."
6,0.95810,"frozenset({Gift_Behavior=No_Gift, Comment_Beha..."
7,0.95810,"frozenset({Gift_Value_Level=No_Gift, Gift_Beha..."
8,0.95355,"frozenset({Like_Behavior=No_Like, Gift_Value_L..."
9,0.95355,"frozenset({Like_Behavior=No_Like, Gift_Behavio..."


In [14]:
rules_2 = association_rules(
    frequent_itemsets_2,
    metric="confidence",
    min_threshold=min_confidence
)

rules_2 = rules_2.sort_values(
    by="lift",
    ascending=False
).reset_index(drop=True)

print("Số luật tạo được:", len(rules_2))

display(
    rules_2[
        [
            "antecedents",
            "consequents",
            "support",
            "confidence",
            "lift"
        ]
    ].head(20).round(4)
)

Số luật tạo được: 405


,antecedents,consequents,support,confidence,lift
0,frozenset({Comment_Behavior=Commented}),frozenset({Watch_Time_Level=Long}),0.0354,0.9834,2.9499
1,frozenset({Like_Behavior=Liked}),frozenset({Watch_Time_Level=Long}),0.0357,0.9261,2.7781
2,"frozenset({Watch_Time_Level=Short, Gift_Behavi...","frozenset({Like_Behavior=No_Like, Gift_Value_L...",0.3330,0.9994,1.0686
3,"frozenset({Gift_Value_Level=No_Gift, Watch_Tim...","frozenset({Like_Behavior=No_Like, Gift_Behavio...",0.3330,0.9994,1.0686
4,"frozenset({Like_Behavior=No_Like, Gift_Value_L...","frozenset({Watch_Time_Level=Short, Gift_Behavi...",0.3330,0.3561,1.0686
5,"frozenset({Like_Behavior=No_Like, Gift_Behavio...","frozenset({Gift_Value_Level=No_Gift, Watch_Tim...",0.3330,0.3561,1.0686
6,"frozenset({Like_Behavior=No_Like, Gift_Value_L...",frozenset({Watch_Time_Level=Short}),0.3330,0.3561,1.0683
7,frozenset({Watch_Time_Level=Short}),"frozenset({Like_Behavior=No_Like, Gift_Value_L...",0.3330,0.9991,1.0683
8,"frozenset({Like_Behavior=No_Like, Gift_Value_L...",frozenset({Watch_Time_Level=Short}),0.3330,0.3561,1.0683
9,frozenset({Watch_Time_Level=Short}),"frozenset({Like_Behavior=No_Like, Gift_Behavio...",0.3330,0.9991,1.0683


## 5. So sánh hai cấu hình

Theo yêu cầu Bài 3, khi chỉ sử dụng một thuật toán cần so sánh ít nhất hai cấu hình ngưỡng.

In [15]:
comparison = pd.DataFrame({
    "Cấu hình": [
        "Cấu hình 1",
        "Cấu hình 2"
    ],
    "Min Support": [
        min_support_1,
        min_support_2
    ],
    "Số Frequent Itemsets": [
        len(frequent_itemsets_1),
        len(frequent_itemsets_2)
    ],
    "Min Confidence": [
        min_confidence,
        min_confidence
    ],
    "Số luật": [
        len(rules_1),
        len(rules_2)
    ]
})

display(comparison)

,Cấu hình,Min Support,Số Frequent Itemsets,Min Confidence,Số luật
0,Cấu hình 1,0.01,103,0.3,544
1,Cấu hình 2,0.03,70,0.3,405


### Nhận xét

Khi tăng `min_support` từ 0.01 lên 0.03, các itemset ít phổ biến bị loại bỏ nên số frequent itemsets và số luật thường giảm. Cấu hình support thấp tạo ra nhiều luật hơn nhưng có thể chứa các luật ít phổ biến; cấu hình support cao tập trung hơn vào các mối quan hệ phổ biến.

## 6. Lọc luật

Giữ các luật có:

- `Lift > 1`
- `Confidence >= 0.30`

Sau đó sắp xếp theo Lift giảm dần để ưu tiên các luật có mức liên kết cao.

In [16]:
rules_filtered = rules_1[
    (rules_1["lift"] > 1) &
    (rules_1["confidence"] >= min_confidence)
].copy()

rules_filtered = rules_filtered.sort_values(
    by=["lift", "confidence"],
    ascending=False
).reset_index(drop=True)

print("Số luật sau khi lọc:", len(rules_filtered))

display(
    rules_filtered[
        [
            "antecedents",
            "consequents",
            "support",
            "confidence",
            "lift"
        ]
    ].head(20).round(4)
)

Số luật sau khi lọc: 409


,antecedents,consequents,support,confidence,lift
0,frozenset({Gift_Value_Level=Low_Gift}),"frozenset({Gift_Behavior=Gifted, Watch_Time_Le...",0.0118,0.9916,70.5788
1,"frozenset({Gift_Behavior=Gifted, Watch_Time_Le...",frozenset({Gift_Value_Level=Low_Gift}),0.0118,0.8434,70.5788
2,"frozenset({Gift_Value_Level=Low_Gift, Watch_Ti...",frozenset({Gift_Behavior=Gifted}),0.0118,1.0000,70.4225
3,frozenset({Gift_Value_Level=Low_Gift}),frozenset({Gift_Behavior=Gifted}),0.0120,1.0000,70.4225
4,frozenset({Gift_Behavior=Gifted}),frozenset({Gift_Value_Level=Low_Gift}),0.0120,0.8415,70.4225
5,frozenset({Gift_Behavior=Gifted}),"frozenset({Gift_Value_Level=Low_Gift, Watch_Ti...",0.0118,0.8345,70.4225
6,"frozenset({Like_Behavior=Liked, Watch_Time_Lev...",frozenset({Comment_Behavior=Commented}),0.0136,0.3810,10.5673
7,frozenset({Comment_Behavior=Commented}),"frozenset({Like_Behavior=Liked, Watch_Time_Lev...",0.0136,0.3773,10.5673
8,"frozenset({Watch_Time_Level=Long, Comment_Beha...",frozenset({Like_Behavior=Liked}),0.0136,0.3836,9.9517
9,frozenset({Like_Behavior=Liked}),"frozenset({Watch_Time_Level=Long, Comment_Beha...",0.0136,0.3528,9.9517


In [17]:
# Loại các luật có item trùng giữa antecedent và consequent
def has_overlap(row):
    return len(
        set(row["antecedents"]) &
        set(row["consequents"])
    ) == 0

rules_final = rules_filtered[
    rules_filtered.apply(has_overlap, axis=1)
].copy()

rules_final = rules_final.sort_values(
    by="lift",
    ascending=False
).reset_index(drop=True)

print("Số luật cuối cùng:", len(rules_final))

display(
    rules_final[
        [
            "antecedents",
            "consequents",
            "support",
            "confidence",
            "lift"
        ]
    ].head(15).round(4)
)

Số luật cuối cùng: 409


,antecedents,consequents,support,confidence,lift
0,frozenset({Gift_Value_Level=Low_Gift}),"frozenset({Gift_Behavior=Gifted, Watch_Time_Le...",0.0118,0.9916,70.5788
1,"frozenset({Gift_Behavior=Gifted, Watch_Time_Le...",frozenset({Gift_Value_Level=Low_Gift}),0.0118,0.8434,70.5788
2,"frozenset({Gift_Value_Level=Low_Gift, Watch_Ti...",frozenset({Gift_Behavior=Gifted}),0.0118,1.0000,70.4225
3,frozenset({Gift_Value_Level=Low_Gift}),frozenset({Gift_Behavior=Gifted}),0.0120,1.0000,70.4225
4,frozenset({Gift_Behavior=Gifted}),frozenset({Gift_Value_Level=Low_Gift}),0.0120,0.8415,70.4225
5,frozenset({Gift_Behavior=Gifted}),"frozenset({Gift_Value_Level=Low_Gift, Watch_Ti...",0.0118,0.8345,70.4225
6,"frozenset({Like_Behavior=Liked, Watch_Time_Lev...",frozenset({Comment_Behavior=Commented}),0.0136,0.3810,10.5673
7,frozenset({Comment_Behavior=Commented}),"frozenset({Like_Behavior=Liked, Watch_Time_Lev...",0.0136,0.3773,10.5673
8,"frozenset({Watch_Time_Level=Long, Comment_Beha...",frozenset({Like_Behavior=Liked}),0.0136,0.3836,9.9517
9,frozenset({Like_Behavior=Liked}),"frozenset({Watch_Time_Level=Long, Comment_Beha...",0.0136,0.3528,9.9517


## 7. Xuất kết quả

Kết quả cuối cùng được lưu để sử dụng trong báo cáo Bài 3.

In [21]:
output_file = "../data/processed/association_rules_kualive.xlsx"

rules_final[
    [
        "antecedents",
        "consequents",
        "support",
        "confidence",
        "lift"
    ]
].to_excel(
    output_file,
    index=False
)

print("Đã xuất kết quả:")
print(output_file)

Đã xuất kết quả:
../data/processed/association_rules_kualive.xlsx


## 8. Tổng kết Bài 3

Sau khi chạy toàn bộ Notebook, sử dụng các số liệu thực tế bên dưới để viết báo cáo kỹ thuật.

In [22]:
print("========== TỔNG KẾT BÀI 3 ==========")

print("Số giao dịch:", len(transactions))
print("Số item:", basket.shape[1])

print("\nCấu hình 1:")
print("Min support:", min_support_1)
print("Frequent itemsets:", len(frequent_itemsets_1))
print("Rules:", len(rules_1))

print("\nCấu hình 2:")
print("Min support:", min_support_2)
print("Frequent itemsets:", len(frequent_itemsets_2))
print("Rules:", len(rules_2))

print("\nSau khi lọc:")
print("Rules:", len(rules_final))

if len(rules_final) > 0:
    print("\nLuật có Lift cao nhất:")
    display(
        rules_final[
            [
                "antecedents",
                "consequents",
                "support",
                "confidence",
                "lift"
            ]
        ].head(5).round(4)
    )
else:
    print("\nKhông có luật nào sau khi lọc.")

========== TỔNG KẾT BÀI 3 ==========
Số giao dịch: 20000
Số item: 12

Cấu hình 1:
Min support: 0.01
Frequent itemsets: 103
Rules: 544

Cấu hình 2:
Min support: 0.03
Frequent itemsets: 70
Rules: 405

Sau khi lọc:
Rules: 409

Luật có Lift cao nhất:


,antecedents,consequents,support,confidence,lift
0,frozenset({Gift_Value_Level=Low_Gift}),"frozenset({Gift_Behavior=Gifted, Watch_Time_Le...",0.0118,0.9916,70.5788
1,"frozenset({Gift_Behavior=Gifted, Watch_Time_Le...",frozenset({Gift_Value_Level=Low_Gift}),0.0118,0.8434,70.5788
2,"frozenset({Gift_Value_Level=Low_Gift, Watch_Ti...",frozenset({Gift_Behavior=Gifted}),0.0118,1.0000,70.4225
3,frozenset({Gift_Value_Level=Low_Gift}),frozenset({Gift_Behavior=Gifted}),0.0120,1.0000,70.4225
4,frozenset({Gift_Behavior=Gifted}),frozenset({Gift_Value_Level=Low_Gift}),0.0120,0.8415,70.4225
